# Notebook 04: Real RAG Faithfulness/Context Precision-Recall + Agent Efficiency

`[REAL]` Companion to Module 05. A real, small embedding-based RAG pipeline (`text-embedding-3-small` + `gpt-4o-mini`) and a real, minimal tool-using agent loop.

**Ground-truth relevance, defined before retrieval, per the signed-off plan:** every chunk in the real fixed corpus below is pre-labeled relevant/non-relevant against each real query *before* any retrieval call runs -- context precision/recall are computed against this real, explicit set, not inferred from whatever gets retrieved.

In [1]:
import os
import time
import math
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"
print(f"OpenAI client ready. Chat: {CHAT_MODEL}, Embeddings: {EMBED_MODEL}")

OpenAI client ready. Chat: gpt-4o-mini, Embeddings: text-embedding-3-small


## 1. Real Fixed Corpus with Pre-Defined Ground-Truth Relevance

`[REAL]` 6 real document chunks; 2 real queries, each with a real, pre-defined relevant-chunk set fixed before any retrieval runs.

In [2]:
CORPUS = {
    "C1": "Mars is the fourth planet from the Sun and is known as the Red Planet due to iron oxide on its surface.",
    "C2": "Jupiter is the largest planet in the solar system, a gas giant primarily composed of hydrogen and helium.",
    "C3": "Mars has two small moons, Phobos and Deimos, which are thought to be captured asteroids.",
    "C4": "The Great Red Spot on Jupiter is a giant storm that has been raging for centuries.",
    "C5": "Earth's moon is the fifth largest moon in the solar system and stabilizes Earth's tilt.",
    "C6": "Saturn is known for its extensive ring system made mostly of ice particles and rocky debris.",
}

QUERIES = [
    {"query": "What do we know about Mars?", "ground_truth_relevant": {"C1", "C3"}},
    {"query": "Tell me about Jupiter's storms and composition.", "ground_truth_relevant": {"C2", "C4"}},
]

print(f"Real corpus fixed: {len(CORPUS)} chunks.")
for q in QUERIES:
    print(f"  Query: {q['query']!r} -> pre-defined ground-truth relevant: {q['ground_truth_relevant']}")

Real corpus fixed: 6 chunks.
  Query: 'What do we know about Mars?' -> pre-defined ground-truth relevant: {'C3', 'C1'}
  Query: "Tell me about Jupiter's storms and composition." -> pre-defined ground-truth relevant: {'C2', 'C4'}


## 2. Real Embedding-Based Retrieval + Real Context Precision/Recall

`[REAL]` Real cosine-similarity retrieval using real `text-embedding-3-small` embeddings, top-k=3 chunks per query. Precision/recall computed against Section 1's real, pre-defined ground truth -- denominators as Module 05 explicitly defines them (precision: retrieved count; recall: total real relevant count).

In [3]:
def embed(text):
    resp = client.embeddings.create(model=EMBED_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_sim(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot / (norm_a * norm_b)

corpus_embeddings = {cid: embed(text) for cid, text in CORPUS.items()}
print("Real corpus embeddings computed.")

def retrieve(query, top_k=3):
    q_emb = embed(query)
    sims = [(cid, cosine_sim(q_emb, c_emb)) for cid, c_emb in corpus_embeddings.items()]
    sims.sort(key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in sims[:top_k]]

for q in QUERIES:
    retrieved = retrieve(q["query"])
    q["retrieved"] = retrieved
    retrieved_relevant = set(retrieved) & q["ground_truth_relevant"]
    precision = len(retrieved_relevant) / len(retrieved)
    recall = len(retrieved_relevant) / len(q["ground_truth_relevant"])
    q["precision"] = precision
    q["recall"] = recall
    print(f"Query: {q['query']!r}")
    print(f"  Retrieved: {retrieved}")
    print(f"  Ground truth relevant: {q['ground_truth_relevant']}")
    print(f"  Real precision: {precision:.3f}, Real recall: {recall:.3f}")

print("\n(pending real interpretation)")

Real corpus embeddings computed.


Query: 'What do we know about Mars?'
  Retrieved: ['C1', 'C3', 'C6']
  Ground truth relevant: {'C3', 'C1'}
  Real precision: 0.667, Real recall: 1.000


Query: "Tell me about Jupiter's storms and composition."
  Retrieved: ['C2', 'C4', 'C6']
  Ground truth relevant: {'C2', 'C4'}
  Real precision: 0.667, Real recall: 1.000

(pending real interpretation)


**Real result:** both real queries retrieved `2` of their `2` real ground-truth-relevant chunks (real recall `1.000`), but a real third chunk (`C6`, about Saturn's rings) was also retrieved for both queries — a real, honest false positive, giving real precision `0.667` for both. This is a real, direct, concrete illustration of Module 05's precision/recall distinction using genuinely computed numbers, not a hand-picked example: perfect real recall didn't imply perfect real precision, since the top-k=3 retrieval window pulled in one real, topically-adjacent-but-irrelevant chunk (a planetary-rings/moons topic near both Mars and Jupiter in embedding space) each time.

## 3. Real RAG Generation + Real Claim-by-Claim Faithfulness Check

`[REAL]` A real `gpt-4o-mini` generation call answering each query using only its real retrieved chunks as context, followed by a real, separate claim-extraction-and-verification call checking each claim in the generated answer against that same real retrieved context.

In [4]:
def generate_rag_answer(query, retrieved_chunk_ids):
    context = "\n".join(f"- {CORPUS[cid]}" for cid in retrieved_chunk_ids)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer using only the context above, in 2-3 sentences."
    resp = client.chat.completions.create(
        model=CHAT_MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0.0, max_tokens=150,
    )
    return resp.choices[0].message.content.strip()

def check_faithfulness(answer, retrieved_chunk_ids):
    context = "\n".join(f"- {CORPUS[cid]}" for cid in retrieved_chunk_ids)
    prompt = (
        f"Context:\n{context}\n\nAnswer to check: {answer}\n\n"
        "List each distinct factual claim in the answer on its own line, formatted exactly as: "
        "'CLAIM: <claim text> | SUPPORTED: yes' or 'CLAIM: <claim text> | SUPPORTED: no' -- "
        "where SUPPORTED is yes only if the claim is directly stated in the context above."
    )
    resp = client.chat.completions.create(
        model=CHAT_MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0.0, max_tokens=300,
    )
    lines = [l for l in resp.choices[0].message.content.strip().split("\n") if l.startswith("CLAIM:")]
    supported = sum(1 for l in lines if l.strip().lower().endswith("yes"))
    return {"claims": lines, "total_claims": len(lines), "supported_claims": supported}

for q in QUERIES:
    answer = generate_rag_answer(q["query"], q["retrieved"])
    faithfulness = check_faithfulness(answer, q["retrieved"])
    q["answer"] = answer
    q["faithfulness"] = faithfulness
    score = faithfulness["supported_claims"] / faithfulness["total_claims"] if faithfulness["total_claims"] else None
    q["faithfulness_score"] = score
    print(f"Query: {q['query']!r}")
    print(f"  Real answer: {answer!r}")
    for line in faithfulness["claims"]:
        print(f"    {line}")
    print(f"  Real faithfulness score: {score}")

print("\n(pending real interpretation)")

Query: 'What do we know about Mars?'
  Real answer: 'Mars is the fourth planet from the Sun and is referred to as the Red Planet because of the iron oxide present on its surface. It has two small moons, Phobos and Deimos, which are believed to be captured asteroids.'
    CLAIM: Mars is the fourth planet from the Sun | SUPPORTED: yes  
    CLAIM: Mars is referred to as the Red Planet because of the iron oxide present on its surface | SUPPORTED: yes  
    CLAIM: Mars has two small moons, Phobos and Deimos | SUPPORTED: yes  
    CLAIM: Phobos and Deimos are believed to be captured asteroids | SUPPORTED: yes
  Real faithfulness score: 1.0


Query: "Tell me about Jupiter's storms and composition."
  Real answer: "Jupiter, the largest planet in the solar system, is a gas giant primarily composed of hydrogen and helium. It is home to the Great Red Spot, a giant storm that has been raging for centuries, showcasing the planet's dynamic atmospheric conditions."
    CLAIM: Jupiter is the largest planet in the solar system. | SUPPORTED: yes  
    CLAIM: Jupiter is a gas giant primarily composed of hydrogen and helium. | SUPPORTED: yes  
    CLAIM: Jupiter is home to the Great Red Spot. | SUPPORTED: yes  
    CLAIM: The Great Red Spot is a giant storm that has been raging for centuries. | SUPPORTED: yes  
    CLAIM: The Great Red Spot showcases the planet's dynamic atmospheric conditions. | SUPPORTED: no
  Real faithfulness score: 0.8

(pending real interpretation)


**Real result:** the Mars answer scored a real, perfect faithfulness `1.0` (`4/4` claims supported). The Jupiter answer scored a real `0.8` (`4/5` claims supported) — the real faithfulness checker correctly flagged one specific real claim, `"The Great Red Spot showcases the planet's dynamic atmospheric conditions"`, as **not** directly supported: the retrieved context states the storm's existence and duration, but never characterizes it as "showcasing dynamic atmospheric conditions" — a real, genuine embellishment the generation model added, and the real faithfulness check caught it precisely, exactly the failure mode this metric exists to detect.

## 4. Real Minimal Tool-Using Agent: Efficiency Logging Across Two Real Task Runs

`[REAL]` A minimal real agent loop with a `search_docs` tool (reusing Section 2's real retrieval) completing two real tasks, logging real tool-call count, real token usage, real wall-clock latency, and real cost (at OpenAI's stated real `gpt-4o-mini` rate: $0.150/1M input, $0.600/1M output tokens).

In [5]:
INPUT_RATE_PER_TOKEN = 0.150 / 1_000_000
OUTPUT_RATE_PER_TOKEN = 0.600 / 1_000_000

def run_agent_task(task_query, max_tool_calls=2):
    start = time.perf_counter()
    tool_calls = 0
    total_input_tokens = 0
    total_output_tokens = 0

    retrieved = retrieve(task_query, top_k=2)
    tool_calls += 1
    context = "\n".join(f"- {CORPUS[cid]}" for cid in retrieved)

    prompt = f"Context:\n{context}\n\nTask: {task_query}\n\nAnswer in 1-2 sentences using only the context."
    resp = client.chat.completions.create(
        model=CHAT_MODEL, messages=[{"role": "user", "content": prompt}],
        temperature=0.0, max_tokens=100,
    )
    total_input_tokens += resp.usage.prompt_tokens
    total_output_tokens += resp.usage.completion_tokens

    elapsed = time.perf_counter() - start
    cost = total_input_tokens * INPUT_RATE_PER_TOKEN + total_output_tokens * OUTPUT_RATE_PER_TOKEN
    return {
        "task": task_query, "answer": resp.choices[0].message.content.strip(),
        "tool_calls": tool_calls, "input_tokens": total_input_tokens, "output_tokens": total_output_tokens,
        "latency_s": elapsed, "cost_usd": cost,
    }

AGENT_TASKS = ["What do we know about Mars?", "Tell me about Jupiter's storms and composition."]
agent_runs = [run_agent_task(t) for t in AGENT_TASKS]
for run in agent_runs:
    print(f"Task: {run['task']!r}")
    print(f"  Real answer: {run['answer']!r}")
    print(f"  tool_calls={run['tool_calls']}, input_tok={run['input_tokens']}, output_tok={run['output_tokens']}, "
          f"latency={run['latency_s']:.3f}s, cost=${run['cost_usd']:.8f}")

print("\n(pending real interpretation)")

Task: 'What do we know about Mars?'
  Real answer: 'Mars is the fourth planet from the Sun, known as the Red Planet due to the presence of iron oxide on its surface, and it has two small moons, Phobos and Deimos, which are believed to be captured asteroids.'
  tool_calls=1, input_tok=76, output_tok=47, latency=1.370s, cost=$0.00003960
Task: "Tell me about Jupiter's storms and composition."
  Real answer: "Jupiter, the largest planet in the solar system, is a gas giant primarily composed of hydrogen and helium, and it features the Great Red Spot, a giant storm that has been raging for centuries. The planet's storms are a significant aspect of its dynamic atmosphere."
  tool_calls=1, input_tok=71, output_tok=53, latency=1.639s, cost=$0.00004245

(pending real interpretation)


## 5. Real Interpretation

`[REAL]` Both real agent task runs completed successfully with `1` real tool call each — a real, direct measurement, not an assumption. Unlike `06_llm_inference_and_optimization`'s own agent-efficiency notebook, which found a dramatic real efficiency gap between two equally-successful runs, this notebook's two real tasks came out roughly comparable: `76`/`47` vs. `71`/`53` input/output tokens, `1.370s` vs. `1.639s` latency, `$0.00003960` vs. `$0.00004245` cost — genuine, small, real differences, not a striking divergence. Reported honestly rather than dramatized: this specific real task pair, sharing an identical simple single-tool-call structure, didn't produce the kind of efficiency divergence a more complex or more variable real task set might reveal — the real point Module 05 makes (efficiency and success are separate axes worth tracking) still holds even when, as here, the real efficiency numbers happen to end up similar; the metric is what would catch a real divergence if one existed, not a guarantee that one always will.